# Intent-to-Treat Effect (ITT) Inference Notebook

This notebook estimates unadjusted assignment-based differences in website visits and conversions in the released Criteo Uplift v2.1 benchmark. It builds on the integrity and balance diagnostics in [01_validation.ipynb](01_validation.ipynb) and follows the estimands defined in the [analysis plan](../ANALYSIS_PLAN.md).

The workflow summarizes the assigned arms, calculates absolute and relative lift, applies two-proportion z-tests, constructs 95% confidence intervals for absolute lift, and checks sampling uncertainty with a binomial bootstrap. `visit` is the primary outcome; `conversion` is a secondary, supporting outcome.

All numerical interpretations below refer to the saved execution results. The analysis retains users in their assigned arms, including treatment-assigned users without recorded exposure. Because the public benchmark was pooled and subsampled, the results describe assignment contrasts in the released sample rather than the original campaigns' population incrementality. See the [data card](../DATA_CARD.md) for those limitations.

In [14]:
from pathlib import Path
import os
import sys

root = Path(r"D:\Projects\A_B_Test\Project")
os.chdir(root)

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.run_sql import run_sql
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import scipy
import statsmodels
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.proportion import confint_proportions_2indep
import seaborn as sns

print('Project Setup')
print("-" * 72)
print(f"{'Working directory':<32}: {Path.cwd()}")

Project Setup
------------------------------------------------------------------------
Working directory               : D:\Projects\A_B_Test\Project


## Hypothesis & Estimand

Let $T$ denote randomized advertising eligibility (`treatment`) and let $Y$ denote a binary outcome. For each outcome, the ITT estimator compares all observations assigned to treatment with all observations assigned to holdout, irrespective of actual ad delivery:

$$
\widehat{\mathrm{ITT}}_Y=\hat p_{T,Y}-\hat p_{C,Y},
\qquad
\hat p_{a,Y}=\frac{\text{outcome-positive observations in arm }a}{n_a}.
$$

Actual `exposure` is determined after assignment and is not randomized. Restricting the comparison to exposed users would change the estimand and could introduce selection bias. The present analysis therefore uses `treatment` throughout.

Absolute lift is the difference in outcome probabilities. Multiplying it by 100 expresses the result in **percentage points**. Relative lift scales that difference by the control rate:

$$
\widehat{\mathrm{REL}}_Y=\frac{\hat p_{T,Y}-\hat p_{C,Y}}{\hat p_{C,Y}}.
$$

The primary hypothesis concerns visits; conversions provide secondary evidence. For each outcome, the code performs a two-sided test at a 5% reference significance level:

$$
H_0:p_{T,Y}-p_{C,Y}=0,
\qquad
H_1:p_{T,Y}-p_{C,Y}\ne0.
$$

A positive estimate favors treatment assignment. The tests and intervals assume independent observations within and across arms under a binomial model. The released sample's processing and unavailable experiment identifiers limit the causal and population-level interpretation of these contrasts.

## Descriptive Statistics

The SQL step generates arm-level metrics and joint assignment/exposure/outcome counts. This section reads `arm_summary.csv`, whose denominators include every observation in each assigned arm.

| Measure | Control (`treatment = 0`) | Treatment (`treatment = 1`) |
| --- | ---: | ---: |
| Observations | 2,096,937 | 11,882,655 |
| Visits | 80,105 | 576,824 |
| Visit rate | 3.820096% | 4.854336% |
| Conversions | 4,063 | 36,711 |
| Conversion rate | 0.193759% | 0.308946% |

The two arms contain **13,979,592 observations** in total, with approximately 85% assigned to treatment. Both observed outcome rates are higher in treatment. Although conversions are rare as a proportion of observations, both arms contain thousands of conversions, supporting the large-sample approximation used below.

These are descriptive released-sample rates. Unequal arm sizes are handled through their separate denominators; raw differences in event counts alone do not measure lift.

In [15]:
run_sql("sql/02_experiment_metrics.sql")
arm_summary = pd.read_csv("results/arm_summary.csv")
print('Descriptive Statistics')
print("-" * 72)
print("Assigned-arm counts")
print(
    arm_summary.loc[:, ["treatment", "n", "n_visit", "n_conversion", "n_exposure"]]
    .assign(treatment=lambda table: table["treatment"].map({0: "Control (0)", 1: "Treatment (1)"}))
    .rename(columns={
        "treatment": "Group", "n": "Users", "n_visit": "Visits",
        "n_conversion": "Conversions", "n_exposure": "Exposed",
    }).to_string(
        index=False,
        formatters={
            "Users": "{:,.0f}".format, "Visits": "{:,.0f}".format,
            "Conversions": "{:,.0f}".format, "Exposed": "{:,.0f}".format,
        },
    )
)
print("\nAssigned-arm rates")
print(
    arm_summary.loc[:, ["treatment", "visit_rate", "conversion_rate", "exposure_rate"]]
    .assign(treatment=lambda table: table["treatment"].map({0: "Control (0)", 1: "Treatment (1)"}))
    .rename(columns={
        "treatment": "Group", "visit_rate": "Visit rate",
        "conversion_rate": "Conversion rate", "exposure_rate": "Exposure rate",
    }).to_string(
        index=False,
        formatters={
            "Visit rate": "{:.6%}".format, "Conversion rate": "{:.6%}".format,
            "Exposure rate": "{:.6%}".format,
        },
    )
)

Count
9
Completed: 02_experiment_metrics.sql
Descriptive Statistics
------------------------------------------------------------------------
Assigned-arm counts
        Group      Users  Visits Conversions Exposed
  Control (0)  2,096,937  80,105       4,063       0
Treatment (1) 11,882,655 576,824      36,711 428,212

Assigned-arm rates
        Group Visit rate Conversion rate Exposure rate
  Control (0)  3.820096%       0.193759%     0.000000%
Treatment (1)  4.854336%       0.308946%     3.603673%


## Effect Size

Absolute and relative lift answer complementary questions. Absolute lift measures the difference in outcome probabilities, while relative lift expresses that difference against the control baseline. Both are reported for visits and conversions.

The code returns proportions as decimal fractions. For interpretation, an absolute difference of `0.01` is **1 percentage point**, whereas a relative lift of `0.25` is **25%**. Keeping these units distinct is especially important for the rare conversion outcome.

### Absolute Lift

For each outcome, subtract the control rate from the treatment rate:

$$
\widehat{\mathrm{ITT}}_Y=\hat p_{T,Y}-\hat p_{C,Y}.
$$

The saved estimates are:

| Outcome | Absolute lift (proportion) | Absolute lift (percentage points) |
| --- | ---: | ---: |
| Visit | 0.0103424031 | +1.034240 |
| Conversion | 0.0011518731 | +0.115187 |

The visit difference is equivalent to approximately **10.34 more visits per 1,000 observations**, and the conversion difference to approximately **1.15 more conversions per 1,000 observations**, when expressing the released-sample rate contrasts on a common scale.

In [16]:
p_c_visit = arm_summary.loc[arm_summary['treatment'] == 0, 'visit_rate'].iloc[0]
p_t_visit = arm_summary.loc[arm_summary['treatment'] == 1, 'visit_rate'].iloc[0]
itt = p_t_visit - p_c_visit
print('Absolute Lift | Visit (Primary)')
print("-" * 72)
print(f"{'Control rate':<32}: {p_c_visit:.6%}")
print(f"{'Treatment rate':<32}: {p_t_visit:.6%}")
print(f"{'Absolute lift (proportion)':<32}: {itt:+.10f}")
print(f"{'Absolute lift (pp)':<32}: {100 * itt:+.6f}")

Absolute Lift | Visit (Primary)
------------------------------------------------------------------------
Control rate                    : 3.820096%
Treatment rate                  : 4.854336%
Absolute lift (proportion)      : +0.0103424031
Absolute lift (pp)              : +1.034240


In [17]:
p_c_con = arm_summary.loc[arm_summary['treatment'] == 0, 'conversion_rate'].iloc[0]
p_t_con = arm_summary.loc[arm_summary['treatment'] == 1, 'conversion_rate'].iloc[0]
itt_con = p_t_con - p_c_con
print('Absolute Lift | Conversion (Secondary)')
print("-" * 72)
print(f"{'Control rate':<32}: {p_c_con:.6%}")
print(f"{'Treatment rate':<32}: {p_t_con:.6%}")
print(f"{'Absolute lift (proportion)':<32}: {itt_con:+.10f}")
print(f"{'Absolute lift (pp)':<32}: {100 * itt_con:+.6f}")

Absolute Lift | Conversion (Secondary)
------------------------------------------------------------------------
Control rate                    : 0.193759%
Treatment rate                  : 0.308946%
Absolute lift (proportion)      : +0.0011518731
Absolute lift (pp)              : +0.115187


### Relative Lift

Relative lift divides the absolute difference by the corresponding control rate:

$$
\widehat{\mathrm{REL}}_Y=\frac{\widehat{\mathrm{ITT}}_Y}{\hat p_{C,Y}}.
$$

| Outcome | Control rate | Treatment rate | Relative lift |
| --- | ---: | ---: | ---: |
| Visit | 3.820096% | 4.854336% | +27.073676% |
| Conversion | 0.193759% | 0.308946% | +59.448812% |

The conversion outcome has the larger relative lift because its control baseline is much smaller. Its absolute difference remains smaller than the visit difference. Relative lift should therefore be read alongside the baseline and percentage-point change, rather than used alone to compare business magnitude. The current code estimates relative lift as a point estimate only; the confidence intervals below concern absolute lift.

In [18]:
rel_itt = itt / p_c_visit
print('Relative Lift | Visit (Primary)')
print("-" * 72)
print(f"{'Relative lift (ratio)':<32}: {rel_itt:+.10f}")
print(f"{'Relative lift (%)':<32}: {rel_itt:+.6%}")

Relative Lift | Visit (Primary)
------------------------------------------------------------------------
Relative lift (ratio)           : +0.2707367554
Relative lift (%)               : +27.073676%


In [19]:

rel_itt_con = itt_con / p_c_con
print('Relative Lift | Conversion (Secondary)')
print("-" * 72)
print(f"{'Relative lift (ratio)':<32}: {rel_itt_con:+.10f}")
print(f"{'Relative lift (%)':<32}: {rel_itt_con:+.6%}")

Relative Lift | Conversion (Secondary)
------------------------------------------------------------------------
Relative lift (ratio)           : +0.5944881177
Relative lift (%)               : +59.448812%


## Hypothesis Testing

The code uses a **two-proportion z-test**, with treatment counts supplied first and control counts second. For each outcome, the null hypothesis is equality of the two proportions. The default two-sided test uses the pooled outcome rate to estimate variance under the null:

$$
\hat p_{\mathrm{pool}}=\frac{x_T+x_C}{n_T+n_C},
\qquad
z=\frac{\hat p_T-\hat p_C}{\sqrt{\hat p_{\mathrm{pool}}(1-\hat p_{\mathrm{pool}})(1/n_T+1/n_C)}}.
$$

| Outcome | z-statistic | Saved two-sided p-value | Outcome-specific conclusion at 5% |
| --- | ---: | ---: | --- |
| Visit | 65.247430 | `0.0` (numerical underflow) | Reject equal visit rates |
| Conversion | 28.516529 | 7.308263e-179 | Reject equal conversion rates |

The visit p-value is too small for the numerical calculation to represent as a positive floating-point value, so it is printed as `0.0`; it is **not mathematically zero**. Both results provide strong evidence against equal proportions under the stated model, and the positive statistics agree with the positive estimated differences.

The primary conclusion concerns visits. The conversion test is supporting, secondary evidence. No multiplicity adjustment is implemented, so these are outcome-specific tests rather than a separately calibrated joint testing procedure. Very small p-values reflect both the observed differences and the large sample; effect sizes and confidence intervals are needed to assess magnitude and precision.

In [20]:
visit_c = arm_summary.loc[arm_summary['treatment'] == 0, 'n_visit'].iloc[0]
visit_t = arm_summary.loc[arm_summary['treatment'] == 1, 'n_visit'].iloc[0]
n_c = arm_summary.loc[arm_summary['treatment'] == 0, 'n'].iloc[0]
n_t = arm_summary.loc[arm_summary['treatment'] == 1, 'n'].iloc[0]
print('Two-Proportion Z-Test | Visit (Primary)')
print("-" * 72)
print(
    pd.DataFrame({
        "Group": ["Treatment", "Control"],
        "Events": [visit_t, visit_c],
        "Users": [n_t, n_c],
    }).to_string(
        index=False,
        formatters={"Events": "{:,.0f}".format, "Users": "{:,.0f}".format},
    )
)

count = [visit_t, visit_c]
nobs = [n_t, n_c]
z, p = proportions_ztest(count, nobs)
print(f"{'z-statistic':<32}: {z:.6f}")
print(
    f"{'Two-sided p-value':<32}: "
    + ("Numerical underflow (reported as 0.0)" if p == 0 else f"{p:.6e}")
)

Two-Proportion Z-Test | Visit (Primary)
------------------------------------------------------------------------
    Group  Events      Users
Treatment 576,824 11,882,655
  Control  80,105  2,096,937
z-statistic                     : 65.247430
Two-sided p-value               : Numerical underflow (reported as 0.0)


In [21]:
con_c = arm_summary.loc[arm_summary['treatment'] == 0, 'n_conversion'].iloc[0]
con_t = arm_summary.loc[arm_summary['treatment'] == 1, 'n_conversion'].iloc[0]
n_c = arm_summary.loc[arm_summary['treatment'] == 0, 'n'].iloc[0]
n_t = arm_summary.loc[arm_summary['treatment'] == 1, 'n'].iloc[0]
print('Two-Proportion Z-Test | Conversion (Secondary)')
print("-" * 72)
print(
    pd.DataFrame({
        "Group": ["Treatment", "Control"],
        "Events": [con_t, con_c],
        "Users": [n_t, n_c],
    }).to_string(
        index=False,
        formatters={"Events": "{:,.0f}".format, "Users": "{:,.0f}".format},
    )
)

count = [con_t, con_c]
nobs = [n_t, n_c]
z_con, p_con = proportions_ztest(count, nobs)
print(f"{'z-statistic':<32}: {z_con:.6f}")
print(
    f"{'Two-sided p-value':<32}: "
    + ("Numerical underflow (reported as 0.0)" if p_con == 0 else f"{p_con:.6e}")
)

Two-Proportion Z-Test | Conversion (Secondary)
------------------------------------------------------------------------
    Group Events      Users
Treatment 36,711 11,882,655
  Control  4,063  2,096,937
z-statistic                     : 28.516529
Two-sided p-value               : 7.308263e-179


## Confidence Intervals

The code constructs **95% confidence intervals for the absolute difference in proportions**, using treatment as group 1 and control as group 2. With `compare="diff"` and no explicit `method`, the installed statsmodels implementation uses the **Newcombe method**, combining the two arms' Wilson score intervals. This is distinct from simply applying the pooled z-test standard error to the estimated difference.

| Outcome | Absolute lift (percentage points) | 95% CI (percentage points) |
| --- | ---: | ---: |
| Visit | +1.034240 | [1.005492, 1.062848] |
| Conversion | +0.115187 | [0.108377, 0.121852] |

Both intervals lie entirely above zero and indicate precise positive differences under the independent-binomial model. In repeated samples satisfying that model, the interval procedure is intended to cover the underlying difference about 95% of the time.

In [22]:
ci = confint_proportions_2indep(
    count1=visit_t,
    nobs1=n_t,
    count2=visit_c,
    nobs2=n_c,
    compare="diff",
    alpha=0.05
)

print('95% Confidence Interval | Visit (Primary)')
print("-" * 72)
print(f"{'Absolute lift (proportion)':<32}: {itt:+.10f}")
print(f"{'Absolute lift (pp)':<32}: {100 * itt:+.6f}")
print(f"{'95% CI (proportion)':<32}: [{ci[0]:.10f}, {ci[1]:.10f}]")
print(f"{'95% CI (pp)':<32}: [{100 * ci[0]:.6f}, {100 * ci[1]:.6f}]")

95% Confidence Interval | Visit (Primary)
------------------------------------------------------------------------
Absolute lift (proportion)      : +0.0103424031
Absolute lift (pp)              : +1.034240
95% CI (proportion)             : [0.0100549236, 0.0106284764]
95% CI (pp)                     : [1.005492, 1.062848]


In [23]:
ci_con = confint_proportions_2indep(
    count1=con_t,
    nobs1=n_t,
    count2=con_c,
    nobs2=n_c,
    compare="diff",
    alpha=0.05
)

print('95% Confidence Interval | Conversion (Secondary)')
print("-" * 72)
print(f"{'Absolute lift (proportion)':<32}: {itt_con:+.10f}")
print(f"{'Absolute lift (pp)':<32}: {100 * itt_con:+.6f}")
print(f"{'95% CI (proportion)':<32}: [{ci_con[0]:.10f}, {ci_con[1]:.10f}]")
print(f"{'95% CI (pp)':<32}: [{100 * ci_con[0]:.6f}, {100 * ci_con[1]:.6f}]")

95% Confidence Interval | Conversion (Secondary)
------------------------------------------------------------------------
Absolute lift (proportion)      : +0.0011518731
Absolute lift (pp)              : +0.115187
95% CI (proportion)             : [0.0010837662, 0.0012185181]
95% CI (pp)                     : [0.108377, 0.121852]


## Bootstrap Validation

The bootstrap checks the sampling variability of the absolute differences using **99,999 replicates**. For each outcome and each assigned arm, it generates a binomial event count using the observed arm size and observed outcome rate, then subtracts the simulated control rate from the simulated treatment rate:

$$
X_T^{*(b)}\sim\mathrm{Binomial}(n_T,\hat p_T),
\qquad
X_C^{*(b)}\sim\mathrm{Binomial}(n_C,\hat p_C),
\qquad
\widehat{\mathrm{ITT}}^{*(b)}=X_T^{*(b)}/n_T-X_C^{*(b)}/n_C.
$$

This is an aggregate binomial bootstrap with fixed arm sizes. For a binary outcome, it gives the same event-count distribution as ordinary within-arm resampling of the observed binary outcomes. It is not a randomization test under the null: each arm is simulated from its own observed rate.

The saved bootstrap summaries are:

| Outcome | Bootstrap mean (percentage points) | Percentile 95% CI (percentage points) | Bootstrap SE (percentage points) |
| --- | ---: | ---: | ---: |
| Visit | 1.034241 | [1.005523, 1.062876] | 0.014652 |
| Conversion | 0.115192 | [0.108421, 0.121954] | 0.003432 |

The bootstrap means closely match the original estimates, and the percentile intervals closely match the analytical intervals. This agreement supports the numerical stability of the uncertainty estimates under the shared independent-binomial assumptions.

The visit bootstrap cell initializes the generator with seed **42**, and the conversion cell reuses that generator. Reproducibility therefore depends on the random generator's execution history: reinitialize it and run the two cells in sequence when comparing runs. The table above transcribes the saved outputs, and small simulation differences can arise when bootstrap cells are rerun.

In [28]:
rng = np.random.default_rng(42)
B = 99999

boot_t = rng.binomial(n_t, visit_t / n_t, B) / n_t
boot_c = rng.binomial(n_c, visit_c / n_c, B) / n_c

boot_itt = boot_t - boot_c

boot_mean = np.mean(boot_itt)
boot_ci = np.quantile(boot_itt, [0.025, 0.975])
boot_se = np.std(boot_itt, ddof=1)
print('Binomial Bootstrap | Visit (Primary)')
print("-" * 72)
print(f"{'Replicates':<32}: {B:,}")
print(f"{'Bootstrap mean (pp)':<32}: {100 * boot_mean:.6f}")
print(f"{'Percentile 95% CI (pp)':<32}: [{100 * boot_ci[0]:.6f}, {100 * boot_ci[1]:.6f}]")
print(f"{'Bootstrap SE (pp)':<32}: {100 * boot_se:.6f}")

Binomial Bootstrap | Visit (Primary)
------------------------------------------------------------------------
Replicates                      : 99,999
Bootstrap mean (pp)             : 1.034241
Percentile 95% CI (pp)          : [1.005523, 1.062876]
Bootstrap SE (pp)               : 0.014652


In [30]:
boot_t_con = rng.binomial(n_t, con_t / n_t, B) / n_t
boot_c_con = rng.binomial(n_c, con_c / n_c, B) / n_c

boot_itt_con = boot_t_con - boot_c_con

boot_mean_con = np.mean(boot_itt_con)
boot_ci_con = np.quantile(boot_itt_con, [0.025, 0.975])
boot_se_con = np.std(boot_itt_con, ddof=1)
print('Binomial Bootstrap | Conversion (Secondary)')
print("-" * 72)
print(f"{'Replicates':<32}: {B:,}")
print(f"{'Bootstrap mean (pp)':<32}: {100 * boot_mean_con:.6f}")
print(f"{'Percentile 95% CI (pp)':<32}: [{100 * boot_ci_con[0]:.6f}, {100 * boot_ci_con[1]:.6f}]")
print(f"{'Bootstrap SE (pp)':<32}: {100 * boot_se_con:.6f}")

Binomial Bootstrap | Conversion (Secondary)
------------------------------------------------------------------------
Replicates                      : 99,999
Bootstrap mean (pp)             : 0.115192
Percentile 95% CI (pp)          : [0.108421, 0.121954]
Bootstrap SE (pp)               : 0.003432


## Statistical Interpretation

**The released benchmark shows a positive and precisely estimated treatment-assignment contrast for visits, with a positive conversion contrast providing secondary support.** Visits increase from **3.820096%** in control to **4.854336%** in treatment: an absolute difference of **1.034240 percentage points** and a relative lift of **27.073676%**. The 95% interval for the absolute visit difference is **[1.005492, 1.062848] percentage points**.

Conversions increase from **0.193759%** to **0.308946%**, an absolute difference of **0.115187 percentage points** and a relative lift of **59.448812%**. The corresponding 95% interval is **[0.108377, 0.121852] percentage points**. The larger relative conversion lift reflects its low baseline and should be considered alongside the smaller absolute difference.

| Evidence | Visit (primary) | Conversion (secondary) |
| --- | --- | --- |
| Estimated absolute difference | +1.034240 percentage points | +0.115187 percentage points |
| Estimated relative lift | +27.073676% | +59.448812% |
| Two-sided z-test | z = 65.247430; p-value underflows to `0.0` | z = 28.516529; p = 7.308263e-179 |
| Analytical 95% CI | [1.005492, 1.062848] percentage points | [0.108377, 0.121852] percentage points |
| Bootstrap percentile 95% CI | [1.005523, 1.062876] percentage points | [0.108421, 0.121954] percentage points |

Both outcome-specific tests reject equality at the 5% reference level. The analytical and bootstrap intervals exclude zero and agree closely, so the direction and estimated precision are consistent across these calculations. The bootstrap shares the same independent-observation assumptions and is not an independent confirmation of causal validity.

**Scope of the conclusion.** These are unadjusted ITT-style contrasts based on assignment in the released, processed sample. Non-uniform negative-label sampling prevents treating the rates and lifts as direct estimates of the original campaigns' population incrementality. Pooled advertiser experiments lack identifiers, preventing experiment-level or cluster-aware analysis; missing stable user identifiers also limit checks of observation independence. Notebook 01's approximately 85%/15% allocation and small marginal balance differences support further analysis, but do not resolve these limitations. No multiplicity adjustment, relative-lift intervals, or heterogeneity analysis is implemented here.

**Implication for subsequent work.** Use these results as the unadjusted reference for power/MDE analysis and covariate adjustment, retaining `treatment` as the assignment variable. A campaign decision additionally requires a practical effect threshold, transportability assumptions, and cost or value information, none of which is established by these p-values. 